# Agentic Workflow

## Learning goals

- understand the difference between a plain pipeline and a stateful agent workflow
- inspect `AgentState` and each workflow node in execution order
- connect architecture documentation to real code in `src/workflow.py`
- compare one example from each supported query type


## Concept explanation

We start by checking the active Python executable and runtime profile again. In this notebook that verification matters even more because we will inspect trace objects, state transitions, and optional dense-retrieval support.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import RuntimeConfig

print(sys.executable)
print(RuntimeConfig.auto_detect())

## What is an agentic workflow

A static pipeline always applies the same recipe. This workflow is still explicit and deterministic, but it can adapt to query type, decide whether local tools are needed, verify grounding, and abstain when the evidence is too weak. The diagram reference for this notebook lives in `docs/architecture.md`.


## Implementation

The rest of this notebook walks through the workflow node by node using the exact code in `src/workflow.py`. Each step shows a state transition or an observable decision boundary.

In [ ]:
import pandas as pd

from src.ingestion import build_demo_index
from src.state import AgentState, create_initial_state
from src.trace_debug import display_trace
from src.workflow import (
    classify_query_node,
    decide_tools_node,
    fallback_or_finalize_node,
    make_plan_node,
    normalize_query_node,
    retrieve_docs_node,
    run_tools_node,
    run_workflow,
    synthesize_answer_node,
    verify_grounding_node,
)

retriever = build_demo_index(persist=False)

## Stateful agent design

`AgentState` is the shared notebook-friendly contract. By looking at it directly, you can see what the workflow knows at every stage and why later nodes have enough context to make conservative decisions.


In [ ]:
state_schema = pd.DataFrame(
    {
        'field': list(AgentState.__annotations__.keys()),
        'type': [str(value) for value in AgentState.__annotations__.values()],
    }
)
state_schema

## Node concept

The first node, `normalize_query`, is intentionally simple. It gives us the smallest visible state transition and sets up the rest of the workflow to operate on a stable, lowercase query string.


In [ ]:
state = create_initial_state('How many days are in the pilot window?')
normalize_query_node(state)
pd.Series({'user_query': state['user_query'], 'normalized_query': state['normalized_query']})

## Query classification

The classifier turns the normalized text into a control signal. In this repository that means one of five query types, plus a `requires_tools` flag that later nodes can inspect.


In [ ]:
classify_query_node(state)
pd.Series({'query_type': state['query_type'], 'requires_tools': state['requires_tools']})

## Planning

Planning here is not hidden chain-of-thought. It is an explicit ordered list of actions the notebook can inspect. This makes the planner easy to explain in interviews and easy to modify in experiments.


In [ ]:
make_plan_node(state)
pd.DataFrame({'planned_step': state['plan']})

## Retrieval

Once the workflow understands the question type, it retrieves the most relevant evidence chunks. We inspect both the chunk ranking and the source documents to see what evidence enters the reasoning context.


In [ ]:
retrieve_docs_node(state, retriever=retriever, top_k=4)
pd.DataFrame(state['retrieved_docs'])[['chunk_id', 'source', 'score', 'text']]

## Tools

The tool stage has two pieces: decide which tools would help, then execute those tools locally. The pilot-window question triggers date reasoning and arithmetic, so the workflow should plan tool use before synthesis.


In [ ]:
decide_tools_node(state)
run_tools_node(state)

pd.DataFrame(state['tool_outputs']) if state['tool_outputs'] else pd.DataFrame([{'message': 'No tool outputs'}])

## Answer synthesis

Synthesis combines retrieved document evidence and tool outputs into a draft answer. The draft is still provisional at this stage because the verifier has not yet judged whether the claims are fully supported.


In [ ]:
synthesize_answer_node(state)
pd.Series({'draft_answer': state['draft_answer'], 'citation_count': len(state['citations'])})

## Verification

The verifier measures support rather than assuming the draft is safe. It looks for coverage and unsupported claims so the workflow can make a cautious finalize-versus-abstain decision.


In [ ]:
verify_grounding_node(state)
pd.Series(state['verification_result'].to_dict())

## Fallback strategy

The final decision node converts evidence quality into product behavior. This is where the workflow prefers abstention over unsupported confidence.


In [ ]:
fallback_or_finalize_node(state)
pd.Series({'final_status': state['final_status'], 'final_answer': state['final_answer']})

## Run workflow

After walking node by node, we should confirm that the public `run_workflow(...)` path behaves the same way. We also use this section to compare one representative question from each supported query type.


In [ ]:
demo_queries = [
    ('simple_lookup', 'What are the main goals of the workspace policy refresh?'),
    ('comparison', 'How is the rollout plan different from the policy refresh?'),
    ('multi_hop', 'How many days are in the pilot window?'),
    ('summary', 'Summarize the loaded documents.'),
    ('insufficient_evidence_risk', 'Who is the current CEO of the company?'),
]
workflow_runs = []
for expected_type, question in demo_queries:
    result = run_workflow(question, retriever=retriever)
    workflow_runs.append(
        {
            'expected_demo_type': expected_type,
            'predicted_type': result['query_type'],
            'requires_tools': result['requires_tools'],
            'final_status': result['final_status'],
            'trace_steps': len(result['trace']),
            'question': question,
            'final_answer': result['final_answer'],
        }
    )

demo_frame = pd.DataFrame(workflow_runs)
demo_frame

## Inspect execution trace

The trace is the fastest way to understand why a run behaved the way it did. Because the SAFE REFACTOR already added node latency, the trace doubles as both a logic view and a lightweight performance view.


In [ ]:
happy_path = run_workflow('How many days are in the pilot window?', retriever=retriever)
display_trace(happy_path['trace'])

## Experiment

Now compare the five query-type demos more directly. The interesting pattern to watch is that insufficient-evidence questions should abstain, while multi-hop questions should take more steps and often use tools.


In [ ]:
demo_frame[['expected_demo_type', 'predicted_type', 'requires_tools', 'final_status', 'trace_steps', 'question']]

## Result analysis

This notebook should leave you with a structural understanding of the workflow: classification decides the mode, planning explains the route, retrieval and tools provide evidence, and verification determines whether the system should answer at all.


In [ ]:
demo_frame.groupby(['predicted_type', 'final_status'])['trace_steps'].mean().reset_index()

## Takeaways

- Agentic behavior in this repository comes from explicit state transitions, not opaque magic.
- Query type, plan shape, and tool usage are all inspectable.
- Trace inspection is part of the workflow design, not an afterthought.
- Abstention is a feature because unsupported answers are more harmful than empty ones.
